In [ ]:
%sql
DROP TABLE IF EXISTS workspace.gold_weather.dim_time;

CREATE TABLE IF NOT EXISTS workspace.gold_weather.dim_time (
  date_key BIGINT,
  date DATE,
  year BIGINT,
  quarter BIGINT,
  month BIGINT,
  month_name STRING,
  day BIGINT,
  day_name STRING,
  week_of_year BIGINT,
  season STRING
)

In [ ]:
import pandas as pd

dates = spark.table("workspace.silver_weather.weather_daily").select("observation_date").toPandas()
date_series = pd.to_datetime(dates["observation_date"]).drop_duplicates().sort_values()

dim = pd.DataFrame({"date": date_series})
dim["date_key"] = dim["date"].dt.strftime("%Y%m%d").astype(int)
dim["year"] = dim["date"].dt.year
dim["quarter"] = dim["date"].dt.quarter
dim["month"] = dim["date"].dt.month
dim["month_name"] = dim["date"].dt.month_name()
dim["day"] = dim["date"].dt.day
dim["day_name"] = dim["date"].dt.day_name()
dim["week_of_year"] = dim["date"].dt.isocalendar().week.astype(int)

# Estaciones del hemisferio sur por mes calendario (aproximacion meteorologica,
# no por equinoccios/solsticios exactos)
season_by_month = {
    12: "verano", 1: "verano", 2: "verano",
    3: "otono", 4: "otono", 5: "otono",
    6: "invierno", 7: "invierno", 8: "invierno",
    9: "primavera", 10: "primavera", 11: "primavera",
}
dim["season"] = dim["month"].map(season_by_month)

# pd.to_datetime deja "date" en datetime64 (Spark lo mapea a TIMESTAMP); la
# tabla declara DATE, asi que hay que bajarlo a date() antes de escribir o
# falla DELTA_FAILED_TO_MERGE_FIELDS al hacer merge de esquema
dim["date"] = dim["date"].dt.date

dim = dim[["date_key", "date", "year", "quarter", "month", "month_name", "day", "day_name", "week_of_year", "season"]]

df_spark = spark.createDataFrame(dim)
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_weather.dim_time")